# 🧠 Deep Dive 01: Understanding Keyword Search (BM25)
### Part 1 of the DocuMind Hybrid RAG Learning Series

Welcome to the first deep-dive lab of **DocuMind**! 

If you come from standard semantic RAG, you are used to generating **dense embeddings** (e.g. 768 or 1536 floating-point numbers) and running cosine similarity. 
While dense embeddings are amazing at understanding general concepts and paraphrases, they **frequently fail in enterprise systems** on:
- **Exact Identifiers**: SKU numbers (`SKU-9021`), policy IDs (`POL-8821`), invoice numbers (`INV-0491`).
- **Error Codes**: `ERR_SSL_PROTOCOL_ERROR_443`, `404_PAGE_NOT_FOUND`.
- **Exact Numbers**: `$4,821,000`, `14.5%`, `Page §4.2(b)`.
- **Specific People / Product Names**: "Dr. Alexander Sterling", "DocuMind v2.1".

In this notebook, we will explore **Keyword Search (BM25)** from the ground up:
1. **The Math of BM25**: Term Frequency Saturation ($k_1$), Inverse Document Frequency (IDF), and Document Length Normalization ($b$).
2. **Hands-on with `rank_bm25`**: How an inverted index works.
3. **Sparse Vectors in Qdrant**: How `FastEmbed` turns text into sparse vector format (`indices` and `values`).
4. **The Failure Case of BM25**: Seeing with your own eyes where BM25 fails and why we need **Hybrid Search**.

In [1]:
# 1. Setup & Imports
import math
import re
from collections import Counter
import pandas as pd
from rank_bm25 import BM25Okapi
from fastembed import SparseTextEmbedding

# Configure pandas for pretty table display
pd.set_option('display.max_colwidth', None)
print("✅ Environment & libraries loaded successfully!")

d:\Abhi\Personal Projects\documind\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment & libraries loaded successfully!


---
## 2. Our Test Corpus: 5 Realistic Enterprise Chunks

Let's create 5 document chunks containing typical enterprise edge cases:
- Chunk 0: Has a specific policy ID `POL-8821` and a refund timeframe.
- Chunk 1: A financial table snippet with exact numbers (`$4,821,000` and `14.2%`).
- Chunk 2: A DevOps technical error snippet (`ERR_SSL_PROTOCOL_ERROR_443`).
- Chunk 3: HR guidelines mentioning specific names ("Dr. Alexander Sterling").
- Chunk 4: General conceptual text on reimbursement that uses completely different words ("per-diem", "disbursements", "repayment").

In [2]:
corpus = [
    "Standard Enterprise Refund Policy: Customers requesting returns under clause POL-8821 must notify customer service within 30 days. No exceptions are granted without executive approval.",
    "Q3 2025 Financial Summary: Total enterprise SaaS revenue reached $4,821,000, representing an increase of 14.2% quarter-over-quarter. Gross margin remained steady at 78%.",
    "Network Diagnostics Log: Production gateway node reported fatal error ERR_SSL_PROTOCOL_ERROR_443 on ingress port 443 during high concurrency traffic spike.",
    "HR Executive Directory: All medical leave accommodations must be authorized by Dr. Alexander Sterling in Employee Health & Wellbeing Services.",
    "Travel & Incidentals Policy: Submission guidelines for employee per-diem allowances, hotel accommodations, and mileage disbursements."
]

df_corpus = pd.DataFrame({"Chunk ID": [f"Chunk {i}" for i in range(len(corpus))], "Text": corpus})
df_corpus

,Chunk ID,Text
0,Chunk 0,Standard Enterprise Refund Policy: Customers requesting returns under clause POL-8821 must notify customer service within 30 days. No exceptions are granted without executive approval.
1,Chunk 1,"Q3 2025 Financial Summary: Total enterprise SaaS revenue reached $4,821,000, representing an increase of 14.2% quarter-over-quarter. Gross margin remained steady at 78%."
2,Chunk 2,Network Diagnostics Log: Production gateway node reported fatal error ERR_SSL_PROTOCOL_ERROR_443 on ingress port 443 during high concurrency traffic spike.
3,Chunk 3,HR Executive Directory: All medical leave accommodations must be authorized by Dr. Alexander Sterling in Employee Health & Wellbeing Services.
4,Chunk 4,"Travel & Incidentals Policy: Submission guidelines for employee per-diem allowances, hotel accommodations, and mileage disbursements."


---
## 3. The Mathematics of BM25 Explained

BM25 stands for **Best Matching 25** (developed by Stephen Robertson, Karen Spärck Jones, and colleagues in the 1990s). It is the undisputed king of keyword retrieval.

Given a Query $Q$ with query terms $q_1, q_2, \dots, q_n$ and a Document $D$, the BM25 score is:

$$\text{Score}(D, Q) = \sum_{q \in Q} \text{IDF}(q) \times \frac{\text{TF}(q, D) \cdot (k_1 + 1)}{\text{TF}(q, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Let's break down the **3 Pillars** of this formula:

### Pillar 1: Inverse Document Frequency (IDF) - Rewarding Rare Words
$$\text{IDF}(q) = \ln\left(\frac{N - n(q) + 0.5}{n(q) + 0.5} + 1\right)$$
- $N$ = Total number of documents in corpus.
- $n(q)$ = Number of documents that contain the word $q$.
- **The Intuition**: If the word is `"the"` or `"policy"`, it appears in almost every chunk, so $\text{IDF} \approx 0$. But if the word is `"POL-8821"` or `"ERR_SSL_PROTOCOL_ERROR_443"`, it appears in only 1 chunk, so $\text{IDF}$ is huge! Rare terms dominate the ranking.

### Pillar 2: Term Frequency (TF) with Saturation ($k_1 = 1.5$)
- In naive TF-IDF, if a document has the word 10 times, it gets 10x the score. This allows "keyword stuffing" to trick search engines.
- BM25 uses a **saturation curve** controlled by $k_1$ (usually $1.2$ to $2.0$). Seeing a term 1 time gives a big boost; seeing it a 2nd time gives a moderate boost; seeing it 20 times plateaus (it saturates). 

### Pillar 3: Document Length Normalization ($b = 0.75$)
- $|D|$ = Length of current document (in tokens).
- $\text{avgdl}$ = Average length of all documents in the corpus.
- **The Intuition**: If a short chunk of 10 words contains the query term, it is likely 100% about that topic. If a 1,000-word chunk mentions the word once in passing, it's probably not the main topic. Parameter $b$ penalizes long documents so short, concise chunks get a fair chance.

In [4]:
# Let's write a simple tokenizer: lowercase and split on words/special tokens
def tokenize(text: str) -> list[str]:
    # Extract words, numbers, and identifiers (e.g., POL-8821, $4,821,000)
    return [token.lower() for token in re.findall(r'[\w\$\-\.\%]+', text)]

tokenized_corpus = [tokenize(doc) for doc in corpus]

print("Sample tokenized chunk 0:")
print(tokenized_corpus[0])

Sample tokenized chunk 0:
['standard', 'enterprise', 'refund', 'policy', 'customers', 'requesting', 'returns', 'under', 'clause', 'pol-8821', 'must', 'notify', 'customer', 'service', 'within', '30', 'days.', 'no', 'exceptions', 'are', 'granted', 'without', 'executive', 'approval.']


---
## 4. Running Keyword Search with `rank_bm25`

In [5]:
# Initialize BM25 on our tokenized corpus
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def search_bm25(query: str, top_k: int = 3):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    
    results = []
    for idx, score in enumerate(scores):
        results.append({
            "Chunk ID": f"Chunk {idx}",
            "BM25 Score": round(score, 4),
            "Text": corpus[idx]
        })
    
    df_results = pd.DataFrame(results).sort_values(by="BM25 Score", ascending=False).reset_index(drop=True)
    print(f"\n🔎 Query: '{query}'")
    print(f"Tokens searched: {tokenized_query}")
    return df_results.head(top_k)

### Test 1: Searching for an Exact Policy ID (`POL-8821`)

In [6]:
search_bm25("What are the rules for POL-8821?")


🔎 Query: 'What are the rules for POL-8821?'
Tokens searched: ['what', 'are', 'the', 'rules', 'for', 'pol-8821']


,Chunk ID,BM25 Score,Text
0,Chunk 0,2.0158,Standard Enterprise Refund Policy: Customers requesting returns under clause POL-8821 must notify customer service within 30 days. No exceptions are granted without executive approval.
1,Chunk 4,1.2701,"Travel & Incidentals Policy: Submission guidelines for employee per-diem allowances, hotel accommodations, and mileage disbursements."
2,Chunk 1,0.0000,"Q3 2025 Financial Summary: Total enterprise SaaS revenue reached $4,821,000, representing an increase of 14.2% quarter-over-quarter. Gross margin remained steady at 78%."


**Notice**: Chunk 0 received a high score, while all other chunks received **0.0**! There is zero hallucination or ambiguity.

### Test 2: Searching for an Exact Number (`$4,821,000`)

In [7]:
search_bm25("Which quarter had revenue of $4,821,000?")


🔎 Query: 'Which quarter had revenue of $4,821,000?'
Tokens searched: ['which', 'quarter', 'had', 'revenue', 'of', '$4', '821', '000']


,Chunk ID,BM25 Score,Text
0,Chunk 1,5.0395,"Q3 2025 Financial Summary: Total enterprise SaaS revenue reached $4,821,000, representing an increase of 14.2% quarter-over-quarter. Gross margin remained steady at 78%."
1,Chunk 0,0.0000,Standard Enterprise Refund Policy: Customers requesting returns under clause POL-8821 must notify customer service within 30 days. No exceptions are granted without executive approval.
2,Chunk 2,0.0000,Network Diagnostics Log: Production gateway node reported fatal error ERR_SSL_PROTOCOL_ERROR_443 on ingress port 443 during high concurrency traffic spike.


### Test 3: Technical Error Code (`ERR_SSL_PROTOCOL_ERROR_443`)

In [8]:
search_bm25("How do I fix ERR_SSL_PROTOCOL_ERROR_443 on port 443?")


🔎 Query: 'How do I fix ERR_SSL_PROTOCOL_ERROR_443 on port 443?'
Tokens searched: ['how', 'do', 'i', 'fix', 'err_ssl_protocol_error_443', 'on', 'port', '443']


,Chunk ID,BM25 Score,Text
0,Chunk 2,4.4956,Network Diagnostics Log: Production gateway node reported fatal error ERR_SSL_PROTOCOL_ERROR_443 on ingress port 443 during high concurrency traffic spike.
1,Chunk 0,0.0000,Standard Enterprise Refund Policy: Customers requesting returns under clause POL-8821 must notify customer service within 30 days. No exceptions are granted without executive approval.
2,Chunk 1,0.0000,"Q3 2025 Financial Summary: Total enterprise SaaS revenue reached $4,821,000, representing an increase of 14.2% quarter-over-quarter. Gross margin remained steady at 78%."


---
## 5. How Qdrant Stores BM25: Sparse Vectors

In traditional vector search, you store a **Dense Vector** (e.g. 768 floating numbers where every single slot has a value).

In Qdrant, BM25 is stored as a **Sparse Vector**:
- Imagine a vocabulary of 30,000 unique words in the English dictionary.
- Chunk 0 only contains ~18 unique words. The remaining 29,982 words are zero!
- Instead of storing 30,000 numbers with 29,982 zeros, Qdrant stores only the **non-zero coordinates**:
  - `indices`: `[token_id_1, token_id_2, token_id_3]`
  - `values`: `[bm25_weight_1, bm25_weight_2, bm25_weight_3]`

Let's see this in action using **FastEmbed** (`Qdrant/bm25`), the official library created by Qdrant:

In [9]:
# Load Qdrant's official BM25 sparse model
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

# Generate sparse embeddings for Chunk 0
chunk_sparse = list(sparse_model.embed([corpus[0]]))[0]

print(f"Original text: {corpus[0][:80]}...")
print(f"\nNumber of non-zero tokens: {len(chunk_sparse.indices)}")
print(f"First 5 token indices: {chunk_sparse.indices[:5]}")
print(f"First 5 BM25 weights:  {[round(v, 3) for v in chunk_sparse.values[:5]]}")

Original text: Standard Enterprise Refund Policy: Customers requesting returns under clause POL...

Number of non-zero tokens: 21
First 5 token indices: [ 964658953  443772008  517146909  203139330 1075114912]
First 5 BM25 weights:  [np.float64(1.597), np.float64(1.597), np.float64(1.597), np.float64(1.597), np.float64(1.851)]


### How Qdrant Computes Sparse Similarity
When a user searches `"POL-8821"`:
1. FastEmbed converts `"POL-8821"` into a query sparse vector: `{indices: [idx_pol], values: [1.8]}`.
2. Qdrant performs an inverted index dot product: it only checks document points that share `idx_pol`!
3. It computes the dot product in **microseconds**, returning the top matches.

---
## 6. The Fatal Flaw of Keyword Search (Why We MUST Have Hybrid!)

Now let's see where BM25 completely crashes.

Let's ask a **conceptual question** where the user uses everyday language instead of the legal/formal terms:

In [12]:
# The user asks about getting their money back for business travel
# Look closely at Chunk 4: it has the answer ('per-diem allowances', 'hotel accommodations', 'mileage disbursements'),
# but it does NOT contain the words 'reimbursed' or 'travel expenses'!

search_bm25("How can an employee get reimbursed for travel expenses?")


🔎 Query: 'How can an employee get reimbursed for travel expenses?'
Tokens searched: ['how', 'can', 'an', 'employee', 'get', 'reimbursed', 'for', 'travel', 'expenses']


,Chunk ID,BM25 Score,Text
0,Chunk 4,2.9291,"Travel & Incidentals Policy: Submission guidelines for employee per-diem allowances, hotel accommodations, and mileage disbursements."
1,Chunk 1,1.0079,"Q3 2025 Financial Summary: Total enterprise SaaS revenue reached $4,821,000, representing an increase of 14.2% quarter-over-quarter. Gross margin remained steady at 78%."
2,Chunk 3,0.3442,HR Executive Directory: All medical leave accommodations must be authorized by Dr. Alexander Sterling in Employee Health & Wellbeing Services.


### 💥 The Failure Analyzed:
- Chunk 4 (the true answer!) got a **0.0 score** or got ranked at the very bottom because it used words like *"per-diem"* and *"disbursements"* instead of *"reimbursed"*!
- Meanwhile, **Dense Semantic Search (Embeddings)** understands that *"reimbursed for travel expenses"* is conceptually synonymous with *"per-diem allowances and mileage disbursements"*.

**Conclusion**:
- **Pure Dense Search**: Wins on concepts and paraphrases, fails on exact IDs, codes, and numbers.
- **Pure BM25 Search**: Wins on exact IDs, codes, and numbers, fails on concepts and vocabulary mismatches.

👉 In **Notebook 02**, we will implement **Hybrid Search** with **Reciprocal Rank Fusion (RRF)** to get the superpower of both combined into one single ranking!